# DTW Validation Data Extraction — Akamine et al. Dataset
**contact: w.pouw@tilburguniversity.edu**


## Steps
1. Extract raw gesture-level time series → one CSV per gesture event

---
## Step 1 — Extract raw hands gesture segments

Trims raw MediaPipe **hands** time series to each annotated gesture window.  
**No smoothing or interpolation** — data stored as-is from the sensor.

**Keypoints:** wrist + 5 finger tips per hand = 12 landmarks × 3 dims = 36 channels.

**Output (`ValidationDataAkamine/processed/`):**
- `timeseries/{comparison_id}_{ts_num}_hands.csv` — one file per gesture event (ts_num 1 or 2)
- `gesture_annotation.csv` — one row per comparison: metadata + Akamine baseline `average_distance`

In [ ]:
import numpy as np
import pandas as pd
import os
from tqdm import tqdm

# both study2 raw time-series folders (study2_1 = pairs 1-20, study2_2 = pairs 21-47)
TS_DIRS  = [
    "../osfstorage-archive (1)/output_timeseries_study2_1/",
    "../osfstorage-archive (1)/output_timeseries_study2_2/",
]
ANNO_CSV = "../github-archive/kinematics/study2/data/elan_annotation/gestural_alignment_processed.csv"
BASE_CSV = "../github-archive/kinematics/study2/data/processed/05_dtw_distance/dtw_distance.csv"
SIZE_CSV = "../github-archive/kinematics/study2/data/processed/05_dtw_distance/gesture_size.csv"
SIM_CSV  = "../github-archive/R/data/similarity_coding_zoom.csv"
OUT_DIR  = "../processed/"
TS_OUT   = OUT_DIR + "timeseries/"
os.makedirs(TS_OUT, exist_ok=True)

# build pair_id → source directory mapping
pair_dir = {}
for ts_dir in TS_DIRS:
    for f in os.listdir(ts_dir):
        if os.path.isdir(os.path.join(ts_dir, f)):
            pid = int(f.split('_')[0])
            pair_dir[pid] = ts_dir

available_pairs = sorted(pair_dir.keys())
anno = pd.read_csv(ANNO_CSV)
anno = anno[anno['pair'].isin(available_pairs)].reset_index(drop=True)
print(f"Available pairs: {available_pairs}")
print(f"Comparisons to process: {len(anno)}")

In [ ]:
# keypoint columns: wrist + 5 finger tips, both hands (36 channels)
sample_file = os.path.join(pair_dir[available_pairs[0]],
                           f"{available_pairs[0]:03d}_a",
                           f"{available_pairs[0]:03d}_a_hands.csv")
all_cols = pd.read_csv(sample_file, nrows=0).columns.tolist()
KEEP    = ['WRIST', 'THUMB_TIP', 'INDEX_FINGER_TIP',
           'MIDDLE_FINGER_TIP', 'RING_FINGER_TIP', 'PINKY_FINGER_TIP']
KP_COLS = [c for c in all_cols if c != 'time' and any(k in c for k in KEEP)]
print(f"Hands keypoint columns: {len(KP_COLS)} ({len(KP_COLS)//3} xyz-triplets)")

In [ ]:
_ts_cache = {}

def load_segment(pair_id: int, speaker: str, begin_ms: float, end_ms: float):
    """Return raw hands gesture segment (frames x KP_COLS) or None."""
    prefix = f"{pair_id:03d}_{speaker.lower()}"
    if prefix not in _ts_cache:
        ts_dir = pair_dir.get(pair_id)
        path   = os.path.join(ts_dir, prefix, f"{prefix}_hands.csv") if ts_dir else None
        if not path or not os.path.exists(path):
            _ts_cache[prefix] = None
        else:
            try:
                _ts_cache[prefix] = pd.read_csv(path, usecols=['time'] + KP_COLS)
            except Exception:
                _ts_cache[prefix] = pd.read_csv(path, usecols=['time'] + KP_COLS,
                                                 engine='python', on_bad_lines='skip')
    ts = _ts_cache[prefix]
    if ts is None:
        return None
    seg = ts[(ts['time'] >= begin_ms) & (ts['time'] <= end_ms)][KP_COLS].copy()
    return seg.reset_index(drop=True) if len(seg) >= 3 else None

In [ ]:
anno_path  = OUT_DIR + 'gesture_annotation.csv'
n_existing = len([f for f in os.listdir(TS_OUT) if f.endswith('_hands.csv')])

if os.path.exists(anno_path) and n_existing > 0:
    print(f"Dataset already exists ({n_existing} files) — skipping extraction.")
else:
    valid_cids, skipped = [], 0
    for _, row in tqdm(anno.iterrows(), total=len(anno), desc="Extracting"):
        s1 = load_segment(row['pair'], row['speaker_1'],
                          row['begin_time_1_adj'], row['end_time_1_adj'])
        s2 = load_segment(row['pair'], row['speaker_2'],
                          row['begin_time_2_adj'], row['end_time_2_adj'])
        if s1 is None or s2 is None:
            skipped += 1; continue
        cid = row['comparison_id']
        s1.to_csv(f"{TS_OUT}{cid}_1_hands.csv", index=False)
        s2.to_csv(f"{TS_OUT}{cid}_2_hands.csv", index=False)
        valid_cids.append(cid)

    # --- build comprehensive annotation ---
    # Use gestural_alignment_processed.csv (anno) as the 1305-row base so no
    # comparisons are dropped (dtw_distance.csv is missing cid 1273).
    base_anno = anno[anno['comparison_id'].isin(set(valid_cids))].reset_index(drop=True)

    # per-keypoint DTW distances — left-join so missing rows get NaN, not dropped.
    # Exclude cols already present in anno (hands_dtw, A_hands, B_hands) to avoid
    # suffix collisions on merge.
    dtw = pd.read_csv(BASE_CSV)
    DTW_SPECIFIC = [c for c in [
        'comparison_id', 'average_distance', 'average_distance_xyz',
        'left_wrist',  'right_wrist',
        'left_thumb',  'right_thumb',
        'left_index',  'right_index',
        'left_middle', 'right_middle',
        'left_ring',   'right_ring',
        'left_pinky',  'right_pinky',
        'left_right_index', 'right_left_index',
    ] if c in dtw.columns]

    # gesture size (amplitude per speaker/hand, full coverage)
    size = pd.read_csv(SIZE_CSV, usecols=[
        'comparison_id', 'average_size',
        'size_A', 'left_size_A', 'right_size_A',
        'size_B', 'left_size_B', 'right_size_B',
    ])

    # human similarity coding: location, handshape, orientation, movement (subset only)
    sim = pd.read_csv(SIM_CSV).rename(columns={'notes': 'sim_notes'})

    df = base_anno.merge(dtw[DTW_SPECIFIC], on='comparison_id', how='left')
    df = df.merge(size,                     on='comparison_id', how='left')
    df = df.merge(sim,                      on='comparison_id', how='left')

    ANNO_COLS = [
        # comparison level
        'comparison_id', 'pair', 'referent',
        'hands_dtw', 'A_hands', 'B_hands',
        # Akamine DTW distances (per keypoint)
        'average_distance', 'average_distance_xyz',
        'left_wrist',  'right_wrist',
        'left_thumb',  'right_thumb',
        'left_index',  'right_index',
        'left_middle', 'right_middle',
        'left_ring',   'right_ring',
        'left_pinky',  'right_pinky',
        'left_right_index', 'right_left_index',
        # gesture size / amplitude
        'average_size',
        'size_A', 'left_size_A', 'right_size_A',
        'size_B', 'left_size_B', 'right_size_B',
        # human similarity coding (100/1305 comparisons rated)
        'location', 'handshape', 'orientation', 'movement', 'sim_notes',
        # gesture 1 features
        'speaker_1', 'round_1', 'director_1', 'target_1', 'accuracy_1',
        'begin_time_1_adj', 'end_time_1_adj', 'duration_1_adj',
        'iconic_1',
        'A_LH_gesture_referent_1', 'A_RH_gesture_referent_1',
        'B_LH_gesture_referent_1', 'B_RH_gesture_referent_1',
        'A_speech_eng_1', 'B_speech_eng_1',
        # gesture 2 features
        'speaker_2', 'round_2', 'director_2', 'target_2', 'accuracy_2',
        'begin_time_2_adj', 'end_time_2_adj', 'duration_2_adj',
        'iconic_2',
        'A_LH_gesture_referent_2', 'A_RH_gesture_referent_2',
        'B_LH_gesture_referent_2', 'B_RH_gesture_referent_2',
        'A_speech_eng_2', 'B_speech_eng_2',
    ]
    ANNO_COLS = [c for c in ANNO_COLS if c in df.columns]
    df[ANNO_COLS].to_csv(anno_path, index=False)

    n_files = len([f for f in os.listdir(TS_OUT) if f.endswith('_hands.csv')])
    seg_mb  = sum(os.path.getsize(TS_OUT+f) for f in os.listdir(TS_OUT)) / 1e6
    print(f"Done. Skipped {skipped} | Saved {len(df)} comparisons")
    print(f"timeseries/ hands files: {n_files}, folder total: {seg_mb:.1f} MB")
    print(f"gesture_annotation.csv: {len(ANNO_COLS)} columns — "
          f"sim_coding coverage: {df['handshape'].notna().sum()}/{len(df)}")

In [ ]:
import matplotlib.pyplot as plt

df_anno = pd.read_csv(anno_path)
print(f"{len(df_anno)} comparisons")
print(df_anno[['comparison_id','pair','referent','hands_dtw','average_distance']].head())

cid  = df_anno['comparison_id'].iloc[0]
info = df_anno.iloc[0]
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for ts_num, ax in zip([1, 2], axes):
    seg = pd.read_csv(f"{TS_OUT}{cid}_{ts_num}_hands.csv")
    for col in ['X_RIGHT_WRIST', 'Y_RIGHT_WRIST',
                'X_RIGHT_INDEX_FINGER_TIP', 'Y_RIGHT_INDEX_FINGER_TIP']:
        if col in seg.columns:
            ax.plot(seg[col].values, label=col.replace('_', ' '))
    speaker = info['speaker_1'] if ts_num == 1 else info['speaker_2']
    ax.set_title(f"Gesture {ts_num} — speaker {speaker} (raw hands)")
    ax.set_xlabel('frame'); ax.legend(fontsize=7)
plt.tight_layout()
plt.show()